In [0]:
%run ../config/config

In [0]:
BASE_NAME_TABLE_OUTPUT = "bcp_tb_score_pd_bhv_pyme_v2"

In [0]:
#Forzar un nombre completamente nuevo para la tabla de salida
OUTPUT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.bcp_tb_score_pd_bhv_pyme_v2_.{ENV}"
print(f"Tabla de salida modificada: {OUTPUT_TABLE}")

In [0]:
import sys
import time
import os
import pickle
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import utils.month_delta import month_delta
import pyspark.sql.functions as F
sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger

spark = SparkSession.builder.getOrCreate()

import mlflow
import mlflow.tracking._model_registry.utils
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks-uc"

In [0]:
# Iniciar Logger
logger = MLOpsLogger(
    name='05_inference',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '05_inference'
    }
)

In [0]:
mlflow.set_registry_uri("databricks")
logger.info(f"MLflow registry URI: {mlflow.get_tracking_uri()}")

In [0]:
if FIRST_RUN and MESES_HISTORICOS_CALIDAD > 0:
    meses_historicos_lista = []
    mes_tmp = MES_VTA
    for i in range(MESES_HISTORICOS_CALIDAD):
        mes_tmp = month_delta(str(mes_tmp), -1)
        meses_historicos_lista.append(mes_tmp)
    meses_historicos_lista.reverse()
    meses_a_procesar = meses_historicos_lista + [MES_VTA]
    logger.info(f"MODO HISTÓRICO: {len(meses_a_procesar)} meses: {meses_a_procesar}")
else:
    meses_a_procesar = [MES_VTA]
    logger.info(f"Procesando únicamente el mes actual: {MES_VTA}")

In [0]:
try:
    logger.log_stage_start('inference_pipeline', MES_VTA=MES_VTA, environment=ENV)
    start_time = time.time()

    # -----------------------------------------------------------
    # 1. CARGAR MODELO (CatBoost pkl)
    # -----------------------------------------------------------
    logger.info(f"Cargando modelo desde {MODEL_FILE_PATH}...")
    if not os.path.exists(MODEL_FILE_PATH):
        raise FileNotFoundError(f"Archivo de modelo no encontrado: {MODEL_FILE_PATH}")

    with open(MODEL_FILE_PATH, "rb") as f:
        model = pickle.load(f)
    expected_features = model.feature_names_
    logger.info(f"✓ Modelo cargado | Árboles: {model.tree_count_} | Features: {len(expected_features)}")

    # -----------------------------------------------------------
    # 2. LEER TABLA MASTER (TABLE_MASTER de config)
    # -----------------------------------------------------------
    logger.info(f"Leyendo tabla master: {TABLE_MASTER}")
    df_master = spark.read.format("delta").table(TABLE_MASTER)

    # Verificar que las features del modelo existen en la tabla
    missing_features = [f for f in expected_features if f not in df_master.columns]
    if missing_features:
        raise ValueError(
            f"Faltan {len(missing_features)} features en {TABLE_MASTER}: {missing_features[:10]}..."
        )
    logger.info(f"✓ Todas las features del modelo están presentes en la tabla")

    # -----------------------------------------------------------
    # 3. DEFINIR UDF DE PREDICCIÓN (CatBoost predict_proba)
    # -----------------------------------------------------------
    @F.pandas_udf("double")
    def predict_catboost(*cols: pd.Series) -> pd.Series:
        pdf = pd.concat(cols, axis=1)
        pdf.columns = expected_features
        if len(pdf) == 0:
            return pd.Series([], dtype="float64")
        preds = model.predict_proba(pdf)[:, 1]
        return pd.Series(preds)

    model_cols = [F.col(col_name).cast("double") for col_name in expected_features]

    # ---------------------------------------------------------------------------
    # 4. ITERAR POR MESES
    # ---------------------------------------------------------------------------
    resultados = []

    for idx_mes, mes_actual in enumerate(meses_a_procesar, 1):
        logger.info("")
        logger.info("=" * 80)
        logger.info(f"SCORING MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual}")
        logger.info("=" * 80)

        mes_start = time.time()

        try:
            # Filtrar mes
            df_mes = df_master.filter(F.col("codmes") == mes_actual)
            count_mes = df_mes.count()

            if count_mes == 0:
                logger.warning(f"⚠️ Sin registros para codmes={mes_actual}, saltando...")
                resultados.append({
                    'mes': mes_actual, 'registros': 0,
                    'exitoso': False, 'error': 'Sin registros'
                })
                continue

            logger.info(f" Registros a scorear: {count_mes:,}")

            # Aplicar modelo
            df_scored = df_mes.withColumn(
                SCORE_NAME_MODEL,  # "PD" de config
                predict_catboost(*model_cols)
            )

            # ---------------------------------------------------------------------------
            # 3.1 APLICAR CALIBRACIÓN (PD_CAL)
            # ---------------------------------------------------------------------------
            # Fórmulas extraídas del notebook del modelador:
            # XB = ln(PD / (1 - PD))
            # XB_CAL = -0.6916726371672072 + 0.8302316543993986 * XB
            # PD_CAL = 1 / (1 + exp(-XB_CAL))
            df_scored = df_scored.withColumn("XB", F.log(F.col(SCORE_NAME_MODEL) / (1 - F.col(SCORE_NAME_MODEL)))) \
                .withColumn("XB_CAL", F.lit(-0.6916726371672072) + F.lit(0.8302316543993986) * F.col("XB")) \
                .withColumn("PD_CAL", 1 / (1 + F.exp(-F.col("XB_CAL"))))
            
            df_scored = df_scored.withColumn("numpd", F.col("PD_CAL"))

            # Debug: limitar filas si está activo
            if DEBUG_MLOPS:
                logger.warning(f" MODO DEBUG: limitando a {DEBUG_ROWS} registros")
                window = Window.orderBy(F.col(CAMPO_PRIMARIO))
                df_scored = df_scored.withColumn("rn", F.row_number().over(window)) \
                    .filter(F.col("rn") <= DEBUG_ROWS).drop("rn")

            # Reparticionar para escritura eficiente
            df_scored = df_scored.repartition(256, F.col(CAMPO_PRIMARIO))

            # Escribir en OUTPUT_TABLE particionado por codmes
            logger.info(f" Escribiendo en: {OUTPUT_TABLE}")

            if spark.catalog.tableExists(OUTPUT_TABLE):
                df_scored.write \
                    .format("delta") \
                    .mode("overwrite") \
                    .option("replaceWhere", f"codmes = {mes_actual}") \
                    .saveAsTable(OUTPUT_TABLE)
            else:
                df_scored.write \
                    .format("delta") \
                    .mode("overwrite") \
                    .partitionBy("codmes") \
                    .saveAsTable(OUTPUT_TABLE)

            mes_duration = time.time() - mes_start
            logger.info(f" ✓ Mes {mes_actual} completado: {count_mes:,} registros en {mes_duration:.2f}s")

            resultados.append({
            'mes': mes_actual, 'registros': count_mes,
            'exitoso': True, 'duracion': mes_duration
            })

        except Exception as e_mes:
            logger.error(f" X Error en mes {mes_actual}: {str(e_mes)}")
            resultados.append({
                'mes': mes_actual, 'registros': 0,
                'exitoso': False, 'error': str(e_mes)
            })
            continue

    # ---------------------------------------------------------------------------
    # 5. RESUMEN FINAL
    # ---------------------------------------------------------------------------
    total_duration = time.time() - start_time
    exitosos = sum(1 for r in resultados if r['exitoso'])
    total_registros = sum(r['registros'] for r in resultados if r['exitoso'])

    logger.info("")
    logger.info("=" * 80)
    logger.info("RESUMEN DE INFERENCIA")
    logger.info("=" * 80)
    logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")
    logger.info(f"Total registros scoreados: {total_registros:,}")

    for r in resultados:
        if r['exitoso']:
            logger.info(f"  ✓ {r['mes']}: {r['registros']:,} registros en {r['duracion']:.2f}s")
        else:
            logger.error(f"  X {r['mes']}: {r.get('error', '?')}")

    # Task values
    dbutils.jobs.taskValues.set(key="inference_completed", value=exitosos > 0)
    dbutils.jobs.taskValues.set(key="total_scored", value=total_registros)
    dbutils.jobs.taskValues.set(key="OUTPUT_TABLE", value=OUTPUT_TABLE)

    logger.log_stage_end('inference_pipeline', duration=total_duration,
                        records_scored=total_registros, meses_ok=exitosos)
    logger.info(f"✓ INFERENCIA COMPLETADA en {total_duration:.2f}s")

    if exitosos == 0:
        raise ValueError("Ningún mes se procesó exitosamente")

except Exception as e:
    logger.log_exception(operation='inference_pipeline', exception=e, should_raise=True)
    raise e

finally:
    if 'logger' in locals():
        logger._flush_all_handlers()
        logger.close()

In [0]:
df_out = spark.read.table(OUTPUT_TABLE)

# Renombrar columna de score a 'numpd' (nombre requerido por LHCL y certificación)
# df_out = df_out.withColumnRenamed("PD_CAL", "numpd")

# df_out.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable(OUTPUT_TABLE)

##Validaciones

In [0]:
from pyspark.sql import functions as F

df_out = spark.read.table(OUTPUT_TABLE)#.filter(F.col("codmes") == 202604)

df_out.select(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("PD").isNull(), 1).otherwise(0)).alias("pd_nulos"),
    F.min("PD").alias("pd_min"), F.max("PD").alias("pd_max"), F.mean("PD").alias("pd_prom"),
    F.min("numpd").alias("numpd_min"), F.max("numpd").alias("numpd_max"),
).show()

In [0]:
from pyspark.sql import functions as F

df_mio = (spark.read.table(OUTPUT_TABLE)
          #.filter(F.col("codmes")==202604)
          .select("codmes","codclaveunicocli", F.col("PD").alias("pd_mio")))

df_she = (spark.table("catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.t06086_pd_bhv_cliente_pyme_202306_202604_32var")
          #.filter(F.col("codmes")==202604)
          .select("codmes","codclaveunicocli", F.col("PD_NUEVO").alias("pd_she")))

print("mio:", df_mio.count(), "| sherly:", df_she.count())

cmp = (df_mio.join(df_she, ["codmes","codclaveunicocli"], "inner")
             .withColumn("dif", F.abs(F.col("pd_mio")-F.col("pd_she"))))

cmp.select(
    F.count("*").alias("n"),
    F.mean("dif").alias("dif_prom"),
    F.max("dif").alias("dif_max"),
    F.sum(F.when(F.col("dif") <= 1e-6, 1).otherwise(0)).alias("coinciden"),
    F.sum(F.when(F.col("dif") <= 1e-4, 1).otherwise(0)).alias("coinciden_1e4"),
).show()

In [0]:
spark.read.table(OUTPUT_TABLE).groupBy("codmes").count().show()